# Z-Image-Turbo｜Kaggle T4 ×2

首次：執行 1～3，再設定 4、執行 5。再次生成：只需修改 4、重跑 5。啟用 GPU T4 ×2 與 Internet。

## 1. 安裝套件（首次）

In [ ]:
# Import the tools for installing ComfyUI.
# 匯入安裝 ComfyUI 所需工具。
import subprocess
import sys
from pathlib import Path

# Use Kaggle's writable workspace.
# 使用 Kaggle 可寫入的工作目錄。
ROOT = Path('/kaggle/working')

# Keep ComfyUI and its extensions across generations.
# 將 ComfyUI 與擴充節點保留在工作目錄中。
COMFY = ROOT / 'ComfyUI'
REPOS = [
    ('https://github.com/Comfy-Org/ComfyUI.git', COMFY),
    ('https://github.com/city96/ComfyUI-GGUF.git', COMFY / 'custom_nodes/ComfyUI-GGUF'),
    ('https://github.com/pollockjj/ComfyUI-MultiGPU.git', COMFY / 'custom_nodes/ComfyUI-MultiGPU'),
]

# Download missing source repositories once.
# 只在缺少原始碼時下載，不重複 clone。
for url, folder in REPOS:
    # Reuse repositories that already exist.
    # 已存在的專案直接沿用。
    if not folder.is_dir():
        # Install the repository and stop if cloning fails.
        # 複製專案，失敗時立即停止。
        subprocess.run(['git', 'clone', '-q', '--depth', '1', url, str(folder)], check=True)

# Avoid reinstalling dependencies when this cell is rerun.
# 重跑此格時避免再次安裝套件。
installed = ROOT / '.zimage_kaggle_installed'

# Install ComfyUI, GGUF, and download dependencies once per session.
# 每個工作階段只安裝一次相關套件。
if not installed.exists():
    # Install the official ComfyUI dependencies.
    # 安裝 ComfyUI 官方依賴套件。
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
    # Install the GGUF extension dependencies.
    # 安裝 GGUF 擴充節點依賴套件。
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt')], check=True)
    # Install helpers used to download models and fetch images.
    # 安裝模型下載與圖片傳輸套件。
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet', 'requests'], check=True)
    # Record successful installation.
    # 記錄本次已完成安裝。
    installed.touch()

# Confirm that installation is ready.
# 顯示安裝完成狀態。
print('ComfyUI installed')

## 2. 下載模型（首次）

In [ ]:
# Import the official Hugging Face downloader.
# 匯入 Hugging Face 官方下載工具。
from huggingface_hub import hf_hub_download

# Preserve the model files and formats from the original notebook.
# 保留原始 Notebook 的模型檔案與格式。
MODEL_FILES = [
    ('unsloth/Z-Image-Turbo-GGUF', 'z-image-turbo-Q6_K.gguf', 'unet/z-image-turbo-Q6_K.gguf'),
    ('Comfy-Org/z_image_turbo', 'split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 'text_encoders/qwen_3_4b_fp8_mixed.safetensors'),
    ('Comfy-Org/z_image_turbo', 'split_files/vae/ae.safetensors', 'vae/ae.safetensors'),
]

# Download each model only if its destination is missing.
# 僅在模型檔案不存在時下載。
for repo, name, relative_path in MODEL_FILES:
    # Resolve the model's ComfyUI destination.
    # 取得 ComfyUI 模型檔案路徑。
    destination = COMFY / 'models' / relative_path
    # Create the destination directory if needed.
    # 如有需要，建立模型目錄。
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Download missing model files without replacing valid ones.
    # 只下載缺少的模型，保留原有檔案。
    if not destination.is_file():
        # Remove broken symlinks left by a previous Kaggle session.
        # 清除前次 Kaggle 工作階段留下的無效連結。
        if destination.is_symlink():
            # Discard the invalid link before recreating it.
            # 移除無效連結，再重新建立。
            destination.unlink()
        # Store a cached copy of the checkpoint on disk.
        # 將模型權重下載至本機快取。
        cached_file = hf_hub_download(repo_id=repo, filename=name)
        # Link the downloaded file into the expected ComfyUI folder.
        # 將已下載模型連結到 ComfyUI 指定目錄。
        destination.symlink_to(cached_file)
    # Show the available model name.
    # 顯示已準備完成的模型名稱。
    print('Ready:', destination.name)

## 3. 啟動模型服務（僅首次）

In [ ]:
# Import the tools for managing a persistent local server.
# 匯入管理常駐服務所需工具。
import time
import requests
import torch

# Require the Kaggle T4 x2 accelerator profile.
# 確認 Kaggle 已啟用雙 GPU。
assert torch.cuda.device_count() >= 2, '請在 Kaggle 設定中選擇 GPU T4 ×2。'

# Keep one server on a dedicated local port.
# 使用固定本機連接埠，沿用同一個 ComfyUI 程序。
BASE = 'http://127.0.0.1:8189'
LOG_PATH = ROOT / 'zimage_server.log'

# Check whether the model server is already running.
# 檢查模型服務是否已經啟動。
try:
    # Probe the ComfyUI status endpoint.
    # 透過 ComfyUI 狀態 API 檢查服務。
    running = requests.get(BASE + '/system_stats', timeout=3).ok
# Treat an unreachable service as stopped.
# 若無法連線，視為尚未啟動。
except requests.RequestException:
    # Request a new server process.
    # 標記需要啟動新程序。
    running = False

# Start ComfyUI only once, with retained model caches.
# 只啟動一次 ComfyUI，並保留模型快取。
if not running:
    # Write server diagnostics to a persistent log file.
    # 將服務執行紀錄寫入 log。
    log_file = LOG_PATH.open('w')
    # Keep models on their assigned GPUs and cache loader results.
    # 讓模型保留在指定 GPU，並快取模型載入結果。
    server = subprocess.Popen(
        [sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1',
         '--port', '8189', '--highvram', '--cache-classic', '--preview-method', 'none'],
        cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    # Close the notebook's handle; the subprocess owns its copy.
    # 關閉 Notebook 的檔案控制代碼，讓子程序繼續寫入。
    log_file.close()

# Wait for ComfyUI to complete startup and register custom nodes.
# 等待 ComfyUI 啟動完成並註冊自訂節點。
for attempt in range(120):
    # Check whether the server responds.
    # 檢查服務是否能正常回應。
    try:
        # Stop waiting when the server is ready.
        # 服務準備完成時停止等待。
        if requests.get(BASE + '/system_stats', timeout=3).ok:
            # Continue to node validation.
            # 接著檢查必要節點。
            break
    # Retry when ComfyUI is still starting.
    # ComfyUI 尚未啟動完成時繼續嘗試。
    except requests.RequestException:
        # Ignore temporary connection errors.
        # 忽略啟動過程中的暫時性連線錯誤。
        pass
    # Avoid excessive status requests while starting.
    # 避免啟動期間過度頻繁查詢。
    time.sleep(2)
# Fail clearly instead of silently continuing with a dead server.
# 若啟動失敗，直接顯示 log 位置。
else:
    # Raise an actionable error message.
    # 顯示可供檢查的 log 檔案。
    raise RuntimeError(f'ComfyUI 啟動失敗，請檢查 {LOG_PATH}')

# Query the registered ComfyUI nodes for both GPU loaders.
# 確認雙 GPU 模型載入節點已成功註冊。
node_info = requests.get(BASE + '/object_info', timeout=60).json()

# Specify the required multi-GPU loader nodes.
# 列出雙 GPU 模型載入需要的節點。
required_nodes = {'UnetLoaderGGUFMultiGPU', 'CLIPLoaderMultiGPU', 'VAELoaderMultiGPU'}

# Detect missing custom nodes before submitting a workflow.
# 在生成前先檢查是否缺少自訂節點。
missing_nodes = required_nodes - node_info.keys()

# Avoid silently falling back to one GPU.
# 缺少節點時直接報錯，不默默退回單 GPU。
assert not missing_nodes, f'找不到雙 GPU 節點：{sorted(missing_nodes)}；請檢查 {LOG_PATH}'

# Show the active GPUs and the retained server connection.
# 顯示 GPU 配置及已啟動的常駐服務。
print('GPU 0:', torch.cuda.get_device_name(0), '｜影像生成')
print('GPU 1:', torch.cuda.get_device_name(1), '｜文字編碼與 VAE')
print('ComfyUI ready:', BASE)

## 4. 生成設定

In [ ]:
# Describe the desired image.
# 設定圖片內容。
PROMPT = 'A tabby cat by a cafe window, warm afternoon light, realistic photography'

# Set the image dimensions and seed.
# 設定圖片尺寸與隨機種子。
WIDTH, HEIGHT, SEED = 640, 960, 42

# Preserve the original Turbo sampling parameters.
# 保留原本的 Turbo 採樣參數。
STEPS, CFG = 8, 1

# Choose the PNG filename; duplicate names get numbered automatically.
# 設定 PNG 檔名；同名時自動編號，避免覆蓋。
OUTPUT_NAME = 'zimage.png'

## 5. 生成圖片（可重複執行）

In [ ]:
# Import tools for tracking and displaying the generated image.
# 匯入追蹤進度與顯示圖片的工具。
import uuid
from IPython.display import FileLink, display
from PIL import Image

# Check image dimensions before submitting a job.
# 檢查圖片尺寸設定是否有效。
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 64 == 0 and HEIGHT % 64 == 0, '寬高須為正整數且為 64 的倍數。'

# Define the same GGUF workflow with its loaders fixed to separate GPUs.
# 保留原本 GGUF 工作流程，並將不同模型固定分配至兩張 GPU。
workflow = {
    '1': {'class_type': 'UnetLoaderGGUFMultiGPU', 'inputs': {'unet_name': 'z-image-turbo-Q6_K.gguf', 'device': 'cuda:0'}},
    '2': {'class_type': 'CLIPLoaderMultiGPU', 'inputs': {'clip_name': 'qwen_3_4b_fp8_mixed.safetensors', 'type': 'lumina2', 'device': 'cuda:1'}},
    '3': {'class_type': 'VAELoaderMultiGPU', 'inputs': {'vae_name': 'ae.safetensors', 'device': 'cuda:1'}},
    '4': {'class_type': 'CLIPTextEncode', 'inputs': {'text': PROMPT, 'clip': ['2', 0]}},
    '5': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['4', 0]}},
    '6': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
    '7': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'shift': 3, 'model': ['1', 0]}},
    '8': {'class_type': 'KSampler', 'inputs': {'seed': SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'res_multistep', 'scheduler': 'simple', 'denoise': 1, 'model': ['7', 0], 'positive': ['4', 0], 'negative': ['5', 0], 'latent_image': ['6', 0]}},
    '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
    '10': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'zimage', 'images': ['9', 0]}},
}

# Submit the image job to the existing ComfyUI server.
# 將生成工作送往已啟動的 ComfyUI，不重新載入模型服務。
response = requests.post(BASE + '/prompt', json={'prompt': workflow, 'client_id': str(uuid.uuid4())}, timeout=120)

# Surface validation errors instead of silently waiting.
# 若工作流程驗證失敗，立即顯示錯誤。
if not response.ok:
    # Show the response body to help diagnose node or setting errors.
    # 顯示伺服器回覆，協助定位節點或設定問題。
    raise RuntimeError(response.text)

# Save the submitted job identifier.
# 記錄本次生成工作的編號。
job_id = response.json()['prompt_id']

# Track the elapsed generation time.
# 記錄本次生成所花時間。
start = time.monotonic()

# Show status while waiting for the PNG output.
# 等待 PNG 輸出時定期顯示進度訊息。
for attempt in range(1800):
    # Read the current workflow history.
    # 讀取目前工作流程的執行紀錄。
    record = requests.get(BASE + '/history/' + job_id, timeout=30).json().get(job_id)
    # Inspect completed or failed jobs.
    # 檢查已完成或失敗的工作。
    if record:
        # Report execution errors from ComfyUI.
        # 顯示 ComfyUI 實際執行錯誤。
        if record.get('status', {}).get('status_str') == 'error':
            # Stop when the server reports an error.
            # 服務回報錯誤時立即停止。
            raise RuntimeError(str(record['status']))
        # Read image metadata when generation finishes.
        # 生成結束後取得圖片資訊。
        images = record.get('outputs', {}).get('10', {}).get('images', [])
        # Stop polling as soon as the image is available.
        # 圖片已可取得時結束等待。
        if images:
            # Continue to downloading the image.
            # 接著下載生成圖片。
            break
    # Print periodic status in the cell output without assuming a step percentage.
    # 定期顯示已等待時間，不捏造採樣百分比。
    if attempt % 5 == 0:
        # Display elapsed seconds while the server is generating.
        # 在 cell 下方顯示目前耗費秒數。
        print(f'生成中… {time.monotonic() - start:.0f} 秒', flush=True)
    # Pause before the next history request.
    # 等待片刻，再次查詢工作狀態。
    time.sleep(2)
# Stop if the server never returns an image.
# 若長時間沒有產生圖片，則回報逾時。
else:
    # Report a clear timeout rather than looping indefinitely.
    # 顯示逾時訊息，避免無限等待。
    raise TimeoutError(f'生成逾時，請檢查 {LOG_PATH}')

# Download the completed PNG from ComfyUI.
# 從 ComfyUI 取得生成完成的 PNG。
image_response = requests.get(BASE + '/view', params=images[0], timeout=120)

# Reject failed downloads to prevent corrupted files.
# 防止下載失敗卻仍儲存不完整圖片。
image_response.raise_for_status()

# Sanitize the requested filename and enforce PNG output.
# 清理輸出檔名並統一儲存為 PNG。
filename = Path(OUTPUT_NAME).name

# Ensure output filenames always have the PNG extension.
# 確保輸出檔名使用 PNG 副檔名。
if not filename.lower().endswith('.png'):
    # Append the standard PNG extension.
    # 自動補上 PNG 副檔名。
    filename += '.png'

# Save results in Kaggle's working directory.
# 將結果儲存在 Kaggle 的工作目錄。
OUTPUT_PATH = ROOT / filename

# Start suffix numbering for duplicate filenames.
# 從 002 開始編號同名圖片。
index = 2

# Preserve earlier generations by choosing an unused filename.
# 選擇未使用的檔名，避免覆蓋先前作品。
while OUTPUT_PATH.exists():
    # Add a numeric suffix for each additional image.
    # 在重複檔名後加入編號。
    OUTPUT_PATH = ROOT / f'{Path(filename).stem}_{index:03d}.png'
    # Move to the next number if this name is already used.
    # 若名稱仍重複，繼續嘗試下一個編號。
    index += 1

# Save the new PNG to Kaggle storage.
# 將生成的 PNG 寫入 Kaggle 儲存空間。
OUTPUT_PATH.write_bytes(image_response.content)

# Show completion time and the saved filename.
# 顯示生成耗時與儲存檔名。
print(f'完成：{OUTPUT_PATH.name}｜{time.monotonic() - start:.1f} 秒')

# Preview the finished image in the notebook.
# 直接在 Notebook 中預覽生成圖片。
display(Image.open(OUTPUT_PATH))

# Provide a local link to the saved PNG.
# 提供 PNG 檔案下載連結。
display(FileLink(str(OUTPUT_PATH)))